<a href="https://colab.research.google.com/github/LAJSol/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Market Stall Performance Analysis

**Client:** Wanjiku, who runs a fresh-produce stall at Gikomba Market, Nairobi.

**Questions to answer:**

(1) How is the business doing?

(2) Is it growing?

(3) What should she restock and where should she focus?

**Approach:** core Python only.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, average, highest and lowest daily sales

**Approach:** `sum()`, `len()`, `max()` and `min()` give the four numbers. `.index()` tells us the position of the best and worst day. Positions 0 to 6 are week 1 and positions 7 to 13 are week 2. The `days` list gives the day name at that same position.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

# Position of the best and worst day in the list
best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)

# Day names come from the days list at the same position
best_day = days[best_pos]
worst_day = days[worst_pos]

# Positions 0-6 are week 1, positions 7-13 are week 2
if best_pos < 7:
    best_week = 1
else:
    best_week = 2

if worst_pos < 7:
    worst_week = 1
else:
    worst_week = 2

print(f"Total sales:   KES {total_sales:,}")
print(f"Average sales: KES {average_sales:,.2f}")
print(f"Highest sales: KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest sales:  KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales:   KES 46,450
Average sales: KES 3,317.86
Highest sales: KES 5,600 (Sat, week 2)
Lowest sales:  KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth

**Approach:** The function `week_totals()` uses slicing. `sales[:7]` is the first seven days (week 1) and `sales[7:]` is the rest (week 2). It returns both totals together, and we unpack them into two variables. Growth is the increase divided by the week 1 total, times 100.

In [3]:
def week_totals(sales):

    """ Return (week 1 total, week 2 total) from a list of 14 daily sales."""

    week1 = sum(sales[:7])
    week2 = sum(sales[7:])
    return week1, week2


week1_total, week2_total = week_totals(daily_sales)

growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.66%


## Task 3: Strong, Normal and Weak days

**Approach:** `classify_day()` compares one day's sales with the average:

- Strong: more than 20% above average (amount is greater than average × 1.20)
- Weak: more than 20% below average (amount is less than average × 0.80)
- Normal: everything in between

We loop through all 14 days, classify each one, and count the results in a dictionary using `.get()`.

In [4]:
""" Return "Strong", "Normal" or "Weak" compared with the average."""
def classify_day(amount, average):

    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    else:
        return "Normal"


strong_limit = average_sales * 1.20
weak_limit = average_sales * 0.80
print(f"Strong means above KES {strong_limit:,.2f}")
print(f"Weak means below KES {weak_limit:,.2f}")
print()

counts = {}

for i in range(len(daily_sales)):
    amount = daily_sales[i]
    day = days[i]
    label = classify_day(amount, average_sales)

    counts[label] = counts.get(label, 0) + 1

    print(f"{day}: KES {amount:,} -> {label}")

print()
print("Number of days in each category:")
print(f"Strong: {counts.get('Strong', 0)}")
print(f"Normal: {counts.get('Normal', 0)}")
print(f"Weak:   {counts.get('Weak', 0)}")

Strong means above KES 3,981.43
Weak means below KES 2,654.29

Mon: KES 2,300 -> Weak
Tue: KES 1,950 -> Weak
Wed: KES 2,800 -> Normal
Thu: KES 2,100 -> Weak
Fri: KES 3,600 -> Normal
Sat: KES 5,200 -> Strong
Sun: KES 4,100 -> Strong
Mon: KES 2,500 -> Weak
Tue: KES 2,200 -> Weak
Wed: KES 3,100 -> Normal
Thu: KES 2,400 -> Weak
Fri: KES 3,900 -> Normal
Sat: KES 5,600 -> Strong
Sun: KES 4,700 -> Strong

Number of days in each category:
Strong: 4
Normal: 4
Weak:   6


## Task 4: Weekdays versus weekends

**Approach:** We go through all 14 days. If the day is Sat or Sun, the sale goes into the `weekend_sales` list. Otherwise it goes into the `weekday_sales` list. Then we take the average of each list and work out how much higher the weekend average is, as a percentage of the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []

for i in range(len(daily_sales)):
    if days[i] == "Sat" or days[i] == "Sun":
        weekend_sales.append(daily_sales[i])
    else:
        weekday_sales.append(daily_sales[i])

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)

difference_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends are {difference_pct:.1f}% higher than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends are 82.5% higher than weekdays


## Task 5: Low stock and gross margins

**Approach:** The function `products_to_reorder()` loops through the `stock` dictionary with `.items()`. If a product's quantity is **at or below** its reorder level (`<=`), its name is added to a list. Then a second loop calculates each product's gross margin: `(price - cost) / price * 100`.

In [6]:
""" Return a list of product names whose qty is at or below the reorder level."""
def products_to_reorder(stock_data):

    low_stock = []
    for name, info in stock_data.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(name)
    return low_stock


reorder_list = products_to_reorder(stock)
print("Products to reorder:", reorder_list)
print()

# Gross margin for each product
margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Gross margin per product:")
for name, margin in margins.items():
    print(f"{name:<10} {margin:.1f}%")

Products to reorder: ['Tomatoes', 'Sukuma']

Gross margin per product:
Tomatoes   33.3%
Onions     30.0%
Sukuma     40.0%
Potatoes   33.3%
Avocado    42.9%


## Task 6: Stall Performance Report

**Approach:** All the results from Tasks 1 to 5 are printed in one report. Money uses `:,` for thousands separators (and `:,.2f` for two decimals). Labels are padded with `:<` so the numbers line up.

In [7]:
line = "--------------------------------------------------"
double_line = "=================================================="

print(double_line)
print("        STALL PERFORMANCE REPORT")
print("      Wanjiku's Stall, Gikomba Market")
print(double_line)

print("SALES SUMMARY (14 days)")
print(line)
print(f"{'Total sales':<22} KES {total_sales:>10,}")
print(f"{'Average per day':<22} KES {average_sales:>10,.2f}")
print(f"{'Best day':<22} KES {highest_sales:>10,}  ({best_day}, week {best_week})")
print(f"{'Worst day':<22} KES {lowest_sales:>10,}  ({worst_day}, week {worst_week})")
print()

print("GROWTH")
print(line)
print(f"{'Week 1 total':<22} KES {week1_total:>10,}")
print(f"{'Week 2 total':<22} KES {week2_total:>10,}")
print(f"{'Week-on-week growth':<22} {growth_pct:>14.2f}%")
print()

print("DAY QUALITY")
print(line)
print(f"{'Strong days':<22} {counts.get('Strong', 0):>14}")
print(f"{'Normal days':<22} {counts.get('Normal', 0):>14}")
print(f"{'Weak days':<22} {counts.get('Weak', 0):>14}")
print()

print("WEEKDAYS VS WEEKENDS")
print(line)
print(f"{'Weekday average':<22} KES {weekday_avg:>10,.2f}")
print(f"{'Weekend average':<22} KES {weekend_avg:>10,.2f}")
print(f"{'Weekend is higher by':<22} {difference_pct:>14.1f}%")
print()

print("STOCK AND MARGINS")
print(line)
print("Product      Qty  Reorder  Margin  Status")
for name, info in stock.items():
    if name in reorder_list:
        status = "REORDER"
    else:
        status = "OK"
    print(f"{name:<10} {info['qty']:>5} {info['reorder']:>8} {margins[name]:>6.1f}%  {status}")
print(double_line)

        STALL PERFORMANCE REPORT
      Wanjiku's Stall, Gikomba Market
SALES SUMMARY (14 days)
--------------------------------------------------
Total sales            KES     46,450
Average per day        KES   3,317.86
Best day               KES      5,600  (Sat, week 2)
Worst day              KES      1,950  (Tue, week 1)

GROWTH
--------------------------------------------------
Week 1 total           KES     22,050
Week 2 total           KES     24,400
Week-on-week growth             10.66%

DAY QUALITY
--------------------------------------------------
Strong days                         4
Normal days                         4
Weak days                           6

WEEKDAYS VS WEEKENDS
--------------------------------------------------
Weekday average        KES   2,685.00
Weekend average        KES   4,900.00
Weekend is higher by             82.5%

STOCK AND MARGINS
--------------------------------------------------
Product      Qty  Reorder  Margin  Status
Tomatoes      18    

## Task 7: Recommendations

The written recommendations are in the final section below.

## Recommendations for Wanjiku

1. **Plan your week around the weekend.** Saturday and Sunday average KES 4,900 a day against KES 2,685 on weekdays (82.5% higher), and all four Strong days were weekends. Stock up on Friday and make sure you are fully supplied on Saturday and Sunday.
2. **Restock Tomatoes and Sukuma immediately.** Tomatoes (18 kg against a 25 kg level) and Sukuma (9 kg against 15 kg) are below their reorder levels. Running out on a weekend would lose sales on your best days.
3. **Push your highest-margin products.** Avocado keeps 42.9% of its price as gross margin and Sukuma keeps 40.0%, against 30.0% for Onions. Display them prominently and keep Avocado in stock (12 kg, just above its 10 kg level).
4. **Work on the weak weekdays.** Six of the 14 days were Weak (below KES 2,654.29), and all six were weekdays. Try a small weekday offer, starting with Tuesday, which was your lowest day at KES 1,950.
5. **The business is growing.** Sales rose 10.66% from KES 22,050 in week 1 to KES 24,400 in week 2. Two weeks is a short record, so keep writing down daily sales and re-run this analysis after another month.